# Multi-IRS uplink deep unfolding: master pipeline

Orchestration only: every cell imports the project modules (`config`, `ao`, `teacher`, `train`, `experiments.*`,
`report`) and calls them. Each code cell is preceded by a markdown cell saying what it does and which artifact it
resumes from, and each can be re-run on its own after Cell 1. Artifacts live under `cfg.root`
(`datasets/`, `checkpoints/`, `logs/`, `results/`); figures (PNG 300 dpi + PDF) and `results_findings.md`
go to `cfg.paper_dir`. `main.tex` is never touched.

Full run (hours):

`E:/payam/env/Scripts/python.exe -m nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 master_pipeline.ipynb`
(run from `E:/payam/code`; use `-m nbconvert`, not `-m jupyter nbconvert`, which can start a different Python's kernel; re-running the same command resumes from the saved artifacts)

Smoke run (tiny config, temporary artifact root and paper dir, never touches the real ones): set the environment
variable `IUF_SMOKE=1`.

## Cell 1: setup, flags, hardware, config

Imports, seeds, hardware/CUDA report, the `Config`, and the flags `RESUME`, `FORCE_RETRAIN`, `FORCE_RERUN`.
Resumes from nothing (this cell only builds the in-memory config).

**`AO_MAX_ITER`** is the iteration cap of the FP-AO baseline (Table I: 100). Setting it to **1000** makes the
1000-iteration AO both the baseline and the teacher: the teacher labels regenerate automatically because their hash
includes `ao_max_iter`, and the training checkpoints and all experiment results then recompute through their own
config hashes / artifact fingerprints. **`TIMING_SUBSET`** is the number of test samples used for the batch-1
timings (`None` = all 1000).

With `IUF_SMOKE=1` the config shrinks to a tiny problem (M = 16, a few hundred samples, 2-point sweeps, short
epochs, CPU training) with `root` and `paper_dir` in a fresh temporary directory.

In [ ]:
import os, sys, time, json, random, re, tempfile, subprocess, dataclasses, warnings
from pathlib import Path

sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

import ao, plotting, report, teacher, train
from config import Config
from experiments import convergence, sweeps

# ---- flags
RESUME = True            # continue training from {name}_last.pt if present
FORCE_RETRAIN = False    # True: retrain from scratch, ignoring checkpoints
FORCE_RERUN = False      # True: recompute every experiment even if its cached result is valid
AO_MAX_ITER = 100        # Table I. 1000 => the 1000-iteration AO is the baseline AND the teacher (see markdown)
TIMING_SUBSET = None     # None = all 1000 test samples for batch-1 timings
RUN_TESTS = os.environ.get("IUF_SKIP_TESTS") != "1"
SMOKE = os.environ.get("IUF_SMOKE") == "1"

# ---- seeds (the pipeline modules use their own fixed seed registry; this only fixes notebook-side randomness)
random.seed(0); np.random.seed(0); torch.manual_seed(0)

# ---- config
if SMOKE:
    _tmp = tempfile.mkdtemp(prefix="iuf_smoke_")
    cfg = Config(M=16, maso_M_train=16, K=6, n_train=200, n_val=50, n_test=50, batch_size=50,
                 n_sup_epochs=2, n_unsup_epochs=2, pilot_sup_epochs=1, pilot_unsup_epochs=1, lr_grid=(0.01, 0.001),
                 ao_max_iter=min(AO_MAX_ITER, 30), timing_subset=3, timing_threads=2, timing_warmup=1,
                 sweep_M=(8, 16), sweep_K=(4, 6), sweep_crn_dbm=(20, 30), zeroshot_M=(8, 24),
                 train_device="cpu", eval_device="cpu",
                 root=os.path.join(_tmp, "artifacts"), paper_dir=os.path.join(_tmp, "Paper"))
    assert os.path.normcase(cfg.root) != os.path.normcase(Config().root)
    assert os.path.normcase(cfg.paper_dir) != os.path.normcase(Config().paper_dir)
    print("SMOKE RUN: tiny config; artifacts in", _tmp)
else:
    cfg = Config(ao_max_iter=AO_MAX_ITER, timing_subset=TIMING_SUBSET)

# ---- hardware report
print("python", sys.version.split()[0], sys.executable, "| torch", torch.__version__, "| cpu threads", torch.get_num_threads(),
      "| cpu count", os.cpu_count())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"CUDA: {p.name}, {p.total_memory / 2**30:.1f} GiB, capability {p.major}.{p.minor}, torch CUDA {torch.version.cuda}")
else:
    print("CUDA: not available (training falls back to CPU; the supplementary GPU timing is skipped)")
print(f"flags: RESUME={RESUME} FORCE_RETRAIN={FORCE_RETRAIN} FORCE_RERUN={FORCE_RERUN} "
      f"AO_MAX_ITER={cfg.ao_max_iter} TIMING_SUBSET={cfg.timing_subset}")
print("root      :", cfg.root)
print("paper_dir :", cfg.paper_dir)
display(pd.Series({k: str(v) for k, v in cfg.to_dict().items()}, name="cfg").to_frame())

# ---- display helpers (presentation only)
def show_png(name):
    png = os.path.join(cfg.paper_dir, f"{name}.png")
    print(png, "|", "pdf ok" if os.path.exists(png.replace(".png", ".pdf")) else "pdf MISSING")
    display(Image(filename=png, width=720))

def show_fig(fig, name):
    """Save PNG (300 dpi) + PDF into cfg.paper_dir, close the figure and show the PNG inline."""
    plotting.save_fig(fig, name, cfg.paper_dir)
    plt.close(fig)
    show_png(name)

def pivot(summary, pattern, row_name="method", col_name="point"):
    """Reshape summary-dict keys matching regex (group 1 = row, group 2 = column) into a DataFrame."""
    rx = re.compile(pattern)
    rec = {}
    for k, v in summary.items():
        m = rx.match(k)
        if m:
            rec.setdefault(m.group(1), {})[int(m.group(2))] = v
    df = pd.DataFrame(rec).T.sort_index(axis=1)
    df.index.name, df.columns.name = row_name, col_name
    return df

def show_keys(summary, prefixes):
    for k in sorted(summary):
        if any(k.startswith(p) for p in prefixes):
            v = summary[k]
            print(f"  {k:48s}", v if isinstance(v, (tuple, str)) else f"{v:.6g}")

## Cell 2: unit tests and the Sec. 5.1 sanity check

Runs `pytest -q tests` in a subprocess (junit XML parsed into a per-file pass/fail table), then
`ao.sanity_check(cfg)`: full AO versus the power-only FP baseline on the validation split (pass iff the gain is
at least 10 %). Resumes from the validation dataset in `datasets/` if present (else it is generated);
`logs/sanity_check.json` is rewritten. Set the environment variable `IUF_SKIP_TESTS=1` to skip the pytest part.

**The cell raises `RuntimeError` (loud banner) if any test fails or the sanity check fails**, so hours of compute are not started on a broken setup. Set the environment variable `IUF_ALLOW_TEST_FAILURE=1` to override and continue anyway.

In [ ]:
import xml.etree.ElementTree as ET

if RUN_TESTS:
    _xml = os.path.join(tempfile.mkdtemp(prefix="iuf_junit_"), "junit.xml")
    _t0 = time.time()
    _proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", f"--junitxml={_xml}"],
                           capture_output=True, text=True)
    rows = {}
    for tc in ET.parse(_xml).getroot().iter("testcase"):
        cn = tc.get("classname", "")
        f = cn.split(".")[-1] if cn else "?"
        r = rows.setdefault(f, {"passed": 0, "failed": 0, "error": 0, "skipped": 0, "time_s": 0.0})
        status = ("failed" if tc.find("failure") is not None else "error" if tc.find("error") is not None
                  else "skipped" if tc.find("skipped") is not None else "passed")
        r[status] += 1
        r["time_s"] += float(tc.get("time", 0))
    tests_df = pd.DataFrame(rows).T
    tests_df.loc["TOTAL"] = tests_df.sum()
    tests_df["status"] = np.where((tests_df["failed"] + tests_df["error"]) == 0, "PASS", "FAIL")
    display(tests_df)
    print(f"pytest return code {_proc.returncode} in {time.time() - _t0:.0f} s")
    TESTS_OK = _proc.returncode == 0
    if not TESTS_OK:
        print(_proc.stdout[-3000:])
else:
    TESTS_OK = True
    print("pytest skipped (IUF_SKIP_TESTS=1)")

sanity = ao.sanity_check(cfg)
display(pd.Series(sanity, name="sanity_check").astype(str).to_frame())
print("Sec. 5.1 sanity check:", "PASSED" if sanity["passed"] else "FAILED", f"(gain {sanity['gain_pct']:.1f} %)")
_problems = [m for ok, m in ((TESTS_OK, "unit tests failed"), (bool(sanity["passed"]), "Sec. 5.1 sanity check failed")) if not ok]
if _problems:
    _msg = " and ".join(_problems)
    print("!" * 70)
    print(f"!!! STOP: {_msg} !!!")
    print("!" * 70)
    if os.environ.get("IUF_ALLOW_TEST_FAILURE") == "1":
        print("IUF_ALLOW_TEST_FAILURE=1: continuing despite the failure")
    else:
        raise RuntimeError(f"{_msg}; fix it or set IUF_ALLOW_TEST_FAILURE=1 to continue anyway")

## Cell 3: datasets and teacher labels

`teacher.build_all_datasets(cfg)` generates (or loads) the train/val/test sets and every sweep-point test set
(fixed seeds); `teacher.build_teacher_labels(cfg)` runs the FP-AO teacher (baseline cap `cfg.ao_max_iter`, plus a
1000-iteration reference on the test split). Resumes from `datasets/*.pt` and `datasets/teacher_labels.pt`
(reused when the file exists and its config hash matches; changing `AO_MAX_ITER` regenerates the labels).

In [ ]:
t0 = time.time()
ds_paths = teacher.build_all_datasets(cfg)
teacher_path = teacher.build_teacher_labels(cfg)
print(f"{len(ds_paths)} datasets ready; teacher labels: {teacher_path} ({time.time() - t0:.0f} s)")
rows = []
for s in ("train", "val", "test"):
    t = teacher.load_teacher(cfg, s)
    rows.append({"split": s, "n": len(t["wsr_ao"]), "mean_WSR_AO": float(t["wsr_ao"].mean()),
                 "std_WSR_AO": float(t["wsr_ao"].std()), "mean_AO_iters": float(t["n_iter"].double().mean()),
                 "pct_hit_cap": 100 * float((t["n_iter"] >= cfg.ao_max_iter).double().mean())})
display(pd.DataFrame(rows).set_index("split"))

## Cell 4: FONR

`train.run_lr_pilot` (learning-rate pilot over `cfg.lr_grid`), `train.train_model` (two-stage training, resumable),
`train.final_metrics` (train/val/test evaluation of `best.pt` in complex128), then the training history table and
inline train/validation curves. Resumes from `logs/fonr_lr_pilot.csv`, `checkpoints/fonr_last.pt` /
`fonr_best.pt` (continues at the next epoch; a finished run is not retrained) and
`logs/fonr_training_history.csv`. `FORCE_RETRAIN = True` discards them.

In [ ]:
NAME = "fonr"
t0 = time.time()
lr = train.run_lr_pilot(NAME, cfg)
best_path = train.train_model(NAME, cfg, resume=RESUME, force_retrain=FORCE_RETRAIN)
fm = train.final_metrics(NAME, cfg)
print(f"[{NAME}] selected lr {lr:g}, best epoch {fm['best_epoch']}, checkpoint {best_path}, {time.time() - t0:.0f} s")

logs = os.path.join(cfg.root, "logs")
pilot = pd.read_csv(os.path.join(logs, f"{NAME}_lr_pilot.csv"))
print("LR pilot: final-epoch mean validation WSR per learning rate")
display(pilot.sort_values("epoch").groupby("lr").tail(1).set_index("lr")[["epoch", "val_wsr_mean"]])

hist = pd.read_csv(os.path.join(logs, f"{NAME}_training_history.csv"))
display(hist.tail(8))
ao_val = float(teacher.load_teacher(cfg, "val")["wsr_ao"].mean())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist["epoch"], hist["train_loss"], label="train loss")
ax[0].plot(hist["epoch"], hist["val_loss"], label="val loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(f"{NAME.upper()}: loss")
ax[1].plot(hist["epoch"], hist["train_wsr_mean"], label="train WSR")
ax[1].plot(hist["epoch"], hist["val_wsr_mean"], label="val WSR")
ax[1].axhline(ao_val, color="k", ls="--", lw=0.8, label=f"FP-AO val mean ({ao_val:.3f})")
for e in hist["epoch"][hist["stage"].ne(hist["stage"].shift()) & (hist.index > 0)]:
    ax[1].axvline(e, color="grey", ls=":", lw=0.8)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("WSR (bits/s/Hz)"); ax[1].legend()
ax[1].set_title(f"{NAME.upper()}: WSR (dotted = stage switch)")
plt.tight_layout(); plt.show()

metrics = pd.DataFrame({s: {"loss": fm[f"{s}_loss"], "WSR mean": fm[f"{s}_wsr_mean"], "WSR std": fm[f"{s}_wsr_std"],
                            "FP-AO WSR mean": fm[f"{s}_ao_wsr_mean"], "efficiency %": fm[f"{s}_efficiency_pct"]}
                        for s in ("train", "val", "test")}).T
display(metrics)
print("test gap to AO (bits/s/Hz): median {:.4g}, 95th pctl {:.4g}, min (left tail) {:.4g}".format(
    fm["test_gap_median"], fm["test_gap_95th_pctl"], fm["test_gap_min_left_tail"]))

## Cell 5: MASO

`train.run_lr_pilot` (learning-rate pilot over `cfg.lr_grid`), `train.train_model` (two-stage training, resumable),
`train.final_metrics` (train/val/test evaluation of `best.pt` in complex128), then the training history table and
inline train/validation curves. Resumes from `logs/maso_lr_pilot.csv`, `checkpoints/maso_last.pt` /
`maso_best.pt` (continues at the next epoch; a finished run is not retrained) and
`logs/maso_training_history.csv`. `FORCE_RETRAIN = True` discards them.

In [ ]:
NAME = "maso"
t0 = time.time()
lr = train.run_lr_pilot(NAME, cfg)
best_path = train.train_model(NAME, cfg, resume=RESUME, force_retrain=FORCE_RETRAIN)
fm = train.final_metrics(NAME, cfg)
print(f"[{NAME}] selected lr {lr:g}, best epoch {fm['best_epoch']}, checkpoint {best_path}, {time.time() - t0:.0f} s")

logs = os.path.join(cfg.root, "logs")
pilot = pd.read_csv(os.path.join(logs, f"{NAME}_lr_pilot.csv"))
print("LR pilot: final-epoch mean validation WSR per learning rate")
display(pilot.sort_values("epoch").groupby("lr").tail(1).set_index("lr")[["epoch", "val_wsr_mean"]])

hist = pd.read_csv(os.path.join(logs, f"{NAME}_training_history.csv"))
display(hist.tail(8))
ao_val = float(teacher.load_teacher(cfg, "val")["wsr_ao"].mean())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist["epoch"], hist["train_loss"], label="train loss")
ax[0].plot(hist["epoch"], hist["val_loss"], label="val loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(f"{NAME.upper()}: loss")
ax[1].plot(hist["epoch"], hist["train_wsr_mean"], label="train WSR")
ax[1].plot(hist["epoch"], hist["val_wsr_mean"], label="val WSR")
ax[1].axhline(ao_val, color="k", ls="--", lw=0.8, label=f"FP-AO val mean ({ao_val:.3f})")
for e in hist["epoch"][hist["stage"].ne(hist["stage"].shift()) & (hist.index > 0)]:
    ax[1].axvline(e, color="grey", ls=":", lw=0.8)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("WSR (bits/s/Hz)"); ax[1].legend()
ax[1].set_title(f"{NAME.upper()}: WSR (dotted = stage switch)")
plt.tight_layout(); plt.show()

metrics = pd.DataFrame({s: {"loss": fm[f"{s}_loss"], "WSR mean": fm[f"{s}_wsr_mean"], "WSR std": fm[f"{s}_wsr_std"],
                            "FP-AO WSR mean": fm[f"{s}_ao_wsr_mean"], "efficiency %": fm[f"{s}_efficiency_pct"]}
                        for s in ("train", "val", "test")}).T
display(metrics)
print("test gap to AO (bits/s/Hz): median {:.4g}, 95th pctl {:.4g}, min (left tail) {:.4g}".format(
    fm["test_gap_median"], fm["test_gap_95th_pctl"], fm["test_gap_min_left_tail"]))

## Cell 6: TIAS

`train.run_lr_pilot` (learning-rate pilot over `cfg.lr_grid`), `train.train_model` (two-stage training, resumable),
`train.final_metrics` (train/val/test evaluation of `best.pt` in complex128), then the training history table and
inline train/validation curves. Resumes from `logs/tias_lr_pilot.csv`, `checkpoints/tias_last.pt` /
`tias_best.pt` (continues at the next epoch; a finished run is not retrained) and
`logs/tias_training_history.csv`. `FORCE_RETRAIN = True` discards them.

In [ ]:
NAME = "tias"
t0 = time.time()
lr = train.run_lr_pilot(NAME, cfg)
best_path = train.train_model(NAME, cfg, resume=RESUME, force_retrain=FORCE_RETRAIN)
fm = train.final_metrics(NAME, cfg)
print(f"[{NAME}] selected lr {lr:g}, best epoch {fm['best_epoch']}, checkpoint {best_path}, {time.time() - t0:.0f} s")

logs = os.path.join(cfg.root, "logs")
pilot = pd.read_csv(os.path.join(logs, f"{NAME}_lr_pilot.csv"))
print("LR pilot: final-epoch mean validation WSR per learning rate")
display(pilot.sort_values("epoch").groupby("lr").tail(1).set_index("lr")[["epoch", "val_wsr_mean"]])

hist = pd.read_csv(os.path.join(logs, f"{NAME}_training_history.csv"))
display(hist.tail(8))
ao_val = float(teacher.load_teacher(cfg, "val")["wsr_ao"].mean())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist["epoch"], hist["train_loss"], label="train loss")
ax[0].plot(hist["epoch"], hist["val_loss"], label="val loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(f"{NAME.upper()}: loss")
ax[1].plot(hist["epoch"], hist["train_wsr_mean"], label="train WSR")
ax[1].plot(hist["epoch"], hist["val_wsr_mean"], label="val WSR")
ax[1].axhline(ao_val, color="k", ls="--", lw=0.8, label=f"FP-AO val mean ({ao_val:.3f})")
for e in hist["epoch"][hist["stage"].ne(hist["stage"].shift()) & (hist.index > 0)]:
    ax[1].axvline(e, color="grey", ls=":", lw=0.8)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("WSR (bits/s/Hz)"); ax[1].legend()
ax[1].set_title(f"{NAME.upper()}: WSR (dotted = stage switch)")
plt.tight_layout(); plt.show()

metrics = pd.DataFrame({s: {"loss": fm[f"{s}_loss"], "WSR mean": fm[f"{s}_wsr_mean"], "WSR std": fm[f"{s}_wsr_std"],
                            "FP-AO WSR mean": fm[f"{s}_ao_wsr_mean"], "efficiency %": fm[f"{s}_efficiency_pct"]}
                        for s in ("train", "val", "test")}).T
display(metrics)
print("test gap to AO (bits/s/Hz): median {:.4g}, 95th pctl {:.4g}, min (left tail) {:.4g}".format(
    fm["test_gap_median"], fm["test_gap_95th_pctl"], fm["test_gap_min_left_tail"]))

## Cell 7: master train/val/test comparison

Reads `logs/all_models_final_metrics.csv` (written by `final_metrics` in Cells 4-6) and the teacher means from
`datasets/teacher_labels.pt`. Rows: FP-AO at the baseline cap (train/val/test means from the teacher labels), the
three models, and the AO-1000 reference on the test split. Saves `logs/master_comparison.csv`.

In [ ]:
fm_all = pd.read_csv(os.path.join(cfg.root, "logs", "all_models_final_metrics.csv")).set_index("model_name")
ao_label = f"FP-AO ({cfg.ao_max_iter} iter)"
T = {s: teacher.load_teacher(cfg, s) for s in ("train", "val", "test")}
rows = {ao_label: {"train WSR": float(T["train"]["wsr_ao"].mean()), "val WSR": float(T["val"]["wsr_ao"].mean()),
                   "test WSR": float(T["test"]["wsr_ao"].mean()), "test efficiency %": 100.0,
                   "test gap median": 0.0, "test gap p95": 0.0}}
for n in ("fonr", "maso", "tias"):
    r = fm_all.loc[n]
    rows[n.upper()] = {"train WSR": r["train_wsr_mean"], "val WSR": r["val_wsr_mean"], "test WSR": r["test_wsr_mean"],
                       "test efficiency %": r["test_efficiency_pct"], "test gap median": r["test_gap_median"],
                       "test gap p95": r["test_gap_95th_pctl"]}
ao1000 = float(T["test"]["wsr_ref1000"].mean())
rows["FP-AO (1000 iter), test reference"] = {"test WSR": ao1000,
                                             "test efficiency %": 100.0 * float(T["test"]["wsr_ao"].mean()) / ao1000}
master = pd.DataFrame(rows).T
master.to_csv(os.path.join(cfg.root, "logs", "master_comparison.csv"))
display(master)
print("model efficiency is relative to the baseline-cap teacher; the last row's efficiency is the baseline-cap AO "
      "relative to AO-1000.")

## Cell 8: Fig. 1 (WSR_vs_iteration)

Mean WSR (95 % CI) versus layer / AO iteration for FONR, MASO, TIAS and FP-AO. Saves `WSR_vs_iteration.png` (300 dpi) and `WSR_vs_iteration.pdf` into `cfg.paper_dir`, shows the PNG, and prints the key numbers.
Resumes from `results/exp_01_convergence.npz`; recomputed only if the file is missing/stale (config hash and
artifact fingerprint) or `FORCE_RERUN = True`.

In [ ]:
conv_npz = convergence.run_convergence(cfg, force_rerun=FORCE_RERUN)
show_fig(convergence.plot_fig1(conv_npz), "WSR_vs_iteration")
conv_sum = convergence.summarize_convergence(conv_npz)
print(f"AO baseline mean {conv_sum['ao100_mean']:.4f}, AO-1000 mean {conv_sum['ao1000_mean']:.4f}")
show_keys(conv_sum, ["recovery_pct_"])

## Cell 9: Fig. 2 (WSR_vs_wall_clock)

Mean WSR versus cumulative CPU wall-clock time (batch 1, complex128, `cfg.timing_threads` threads). Saves `WSR_vs_wall_clock.png` (300 dpi) and `WSR_vs_wall_clock.pdf` into `cfg.paper_dir`, shows the PNG, and prints the key numbers.
Resumes from `results/exp_01_convergence.npz`; recomputed only if the file is missing/stale (config hash and
artifact fingerprint) or `FORCE_RERUN = True`.

In [ ]:
conv_npz = convergence.run_convergence(cfg, force_rerun=FORCE_RERUN)
show_fig(convergence.plot_fig2(conv_npz), "WSR_vs_wall_clock")
conv_sum = convergence.summarize_convergence(conv_npz)
show_keys(conv_sum, ["time_to_", "ao_conv_time_ms_", "ao_conv_iters_mean_", "layer_time_ms_"])

## Cell 10: Fig. 3 (CDF)

Empirical CDF of the per-sample WSR gap to the converged AO on the test split, derived by `convergence.run_cdf`
from the convergence result. Saves `CDF.png` / `CDF.pdf` into `cfg.paper_dir`, shows the PNG, prints gap statistics.
Resumes from `results/exp_03_cdf.npz` (rebuilt from `exp_01_convergence.npz` if missing or stale) or
`FORCE_RERUN = True`.

In [ ]:
cdf_npz = convergence.run_cdf(cfg, force_rerun=FORCE_RERUN)
show_fig(convergence.plot_fig3(cdf_npz), "CDF")
conv_sum = convergence.summarize_convergence(convergence.run_convergence(cfg))
show_keys(conv_sum, ["gap_"])

## Cell 11: Fig. 4 (avg_time_vs_Number_of_RIS_M)

Average CPU time per iteration (AO) / per layer (nets) versus the number of IRS elements M (batch 1, complex128; MASO instantiated untrained at each M). Saves `avg_time_vs_Number_of_RIS_M.png` / `avg_time_vs_Number_of_RIS_M.pdf` into `cfg.paper_dir`, shows the PNG, prints the key numbers.
Resumes from `results/exp_04_runtime_vs_M.npz`; recomputed only if the file is missing/stale (config hash and artifact
fingerprint) or `FORCE_RERUN = True`.

In [ ]:
npz_path, summ = sweeps.run_runtime_vs_M(cfg, force_rerun=FORCE_RERUN)
show_png("avg_time_vs_Number_of_RIS_M")
print("result file:", npz_path)
display(pivot(summ, r"^runtime_ms_(.+)_M(\d+)$", "method (mean ms)", "M"))

## Cell 12: Fig. 5 (etha_vs_CRN_transmit_power)

Efficiency eta = WSR(model) / WSR(FP-AO) versus CRN peak power (PMN = CRN - 10 dB), models trained at 20 dBm. Saves `etha_vs_CRN_transmit_power.png` / `etha_vs_CRN_transmit_power.pdf` into `cfg.paper_dir`, shows the PNG, prints the key numbers.
Resumes from `results/exp_05_power_sweep.npz`; recomputed only if the file is missing/stale (config hash and artifact
fingerprint) or `FORCE_RERUN = True`.

In [ ]:
npz_path, summ = sweeps.run_power_sweep(cfg, force_rerun=FORCE_RERUN)
show_png("etha_vs_CRN_transmit_power")
print("result file:", npz_path)
display(pivot(summ, r"^eta_ratio_of_means_(.+)_P(\d+)$", "eta % (ratio of means)", "CRN dBm"))
print("eta range per model (min, max):", {n: summ[f"eta_range_{n}"] for n in ("fonr", "maso", "tias")})

## Cell 13: Fig. 6 (WSR_vs_Number_of_active_users_K)

Mean WSR versus the number of active users K for FP-AO and the three trained models (no retraining). Saves `WSR_vs_Number_of_active_users_K.png` / `WSR_vs_Number_of_active_users_K.pdf` into `cfg.paper_dir`, shows the PNG, prints the key numbers.
Resumes from `results/exp_06_wsr_vs_K.npz`; recomputed only if the file is missing/stale (config hash and artifact
fingerprint) or `FORCE_RERUN = True`.

In [ ]:
npz_path, summ = sweeps.run_wsr_vs_K(cfg, force_rerun=FORCE_RERUN)
show_png("WSR_vs_Number_of_active_users_K")
print("result file:", npz_path)
display(pivot(summ, r"^wsr_(.+)_K(\d+)$", "mean WSR", "K"))
display(pivot(summ, r"^eta_(.+)_K(\d+)$", "eta % vs FP-AO", "K"))

## Cell 14: Fig. 7 (avg_time_vs_Number_of_active_users_K)

Average CPU time per iteration / layer versus K (M = cfg.M, batch 1, complex128). Saves `avg_time_vs_Number_of_active_users_K.png` / `avg_time_vs_Number_of_active_users_K.pdf` into `cfg.paper_dir`, shows the PNG, prints the key numbers.
Resumes from `results/exp_07_runtime_vs_K.npz`; recomputed only if the file is missing/stale (config hash and artifact
fingerprint) or `FORCE_RERUN = True`.

In [ ]:
npz_path, summ = sweeps.run_runtime_vs_K(cfg, force_rerun=FORCE_RERUN)
show_png("avg_time_vs_Number_of_active_users_K")
print("result file:", npz_path)
display(pivot(summ, r"^runtime_ms_(.+)_K(\d+)$", "method (mean ms)", "K"))

## Cell 15: Fig. 8 (WSR_vs_M_ZeroShot)

Zero-shot mean WSR versus M with the models trained at M = cfg.M (MASO zero-pads / truncates). Saves `WSR_vs_M_ZeroShot.png` / `WSR_vs_M_ZeroShot.pdf` into `cfg.paper_dir`, shows the PNG, prints the key numbers.
Resumes from `results/exp_08_zeroshot_M.npz`; recomputed only if the file is missing/stale (config hash and artifact
fingerprint) or `FORCE_RERUN = True`.

In [ ]:
npz_path, summ = sweeps.run_zeroshot_M(cfg, force_rerun=FORCE_RERUN)
show_png("WSR_vs_M_ZeroShot")
print("result file:", npz_path)
display(pivot(summ, r"^wsr_(.+)_M(\d+)_zeroshot$", "mean WSR", "M"))
display(pivot(summ, r"^eta_(.+)_M(\d+)_zeroshot$", "eta % vs FP-AO", "M"))
print("training M (reference):", summ["M_reference"])

## Cell 16: supplementary GPU timing and the findings report

`sweeps.run_gpu_timing(cfg)` times every method on CUDA in complex64 (skipped with a message when no GPU is
available), then `report.build_report(cfg)` rebuilds all summaries from the cached result files and writes
`results_findings.md` (claims vs. measured, deviations, metrics, latency, LR pilot, eta, suggested text changes)
into `cfg.paper_dir`. Resumes from `results/exp_supp_gpu_timing.npz` and every other `results/*.npz`.
`main.tex` is not modified.

In [ ]:
gpu_path, gpu_summ = sweeps.run_gpu_timing(cfg, force_rerun=FORCE_RERUN)
print("GPU timing:", gpu_path if gpu_path else "skipped (no CUDA)")
if gpu_path:
    show_keys(gpu_summ, ["step_time_ms_", "ao_conv_time_ms_"])

report_path = report.build_report(cfg)
print("report written to:", report_path)
text = Path(report_path).read_text(encoding="utf-8")
head = text.split("## 1. Claims vs. measured", 1)[1].split("###", 1)[0].strip().splitlines()
print(f"report: {len(text.splitlines())} lines; head of the claims section follows (first 14 lines)")
display(Markdown("\n".join(head[:14])))